<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo021_Limpieza_encadenada_paso_a_paso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 21 — Limpieza encadenada paso a paso

## Breve repaso

En los capítulos anteriores trabajamos distintas tareas de preparación de datos por separado. Analizamos valores faltantes, estudiamos estrategias para tratarlos, revisamos duplicados, limpiamos textos y categorías, convertimos tipos de datos, trabajamos con fechas, estandarizamos nombres de columnas y creamos nuevas variables útiles para el análisis.

Cada una de esas tareas es importante, pero en un trabajo real rara vez aparecen completamente separadas. Cuando recibimos un dataset, los problemas suelen estar mezclados: una columna puede tener un nombre poco cómodo, estar cargada como texto, contener valores faltantes y además incluir códigos como `"UNKNOWN"` o `"ERROR"`.

Por eso, en este capítulo vamos a construir un flujo de limpieza paso a paso.

La idea no será presentar herramientas nuevas, sino integrar varias herramientas ya trabajadas dentro de una secuencia razonable. Vamos a partir de un dataset crudo, aplicar transformaciones en orden, verificar resultados intermedios y construir una versión más preparada para analizar.

Este tipo de flujo suele llamarse pipeline de limpieza o proceso de preparación de datos. No significa que exista una única receta válida para todos los datasets. Significa que organizamos las transformaciones de manera clara, repetible y verificable.

Al finalizar este capítulo deberías poder:

- Comprender por qué conviene ordenar las tareas de limpieza.
- Construir una copia de trabajo a partir de un dataset crudo.
- Estandarizar nombres de columnas como primer paso estructural.
- Convertir columnas numéricas y temporales.
- Tratar valores problemáticos en columnas categóricas.
- Crear columnas calculadas y de validación.
- Revisar el estado del dataset después de cada etapa.
- Construir una versión final preparada para análisis.

## Dataset de trabajo

Para construir este flujo de limpieza vamos a volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset es adecuado para integrar varias tareas porque contiene distintos tipos de problemas: columnas con nombres poco cómodos para programar, valores faltantes, columnas numéricas cargadas como texto, fechas que necesitan conversión y valores categóricos problemáticos como `"UNKNOWN"` o `"ERROR"`.

En este capítulo vamos a trabajar sobre una copia del dataset original. La idea será conservar el punto de partida y construir, paso a paso, una versión preparada para el análisis.

In [43]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df_original = pd.read_csv(ruta_csv)

df_original.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


El `DataFrame` `df_original` representa el dataset tal como fue cargado desde el archivo.

Antes de transformarlo, conviene hacer una inspección general. No vamos a explicar nuevamente cada herramienta, porque ya las usamos en capítulos anteriores. Aquí las utilizamos como parte de una rutina inicial de diagnóstico.

In [44]:
df_original.shape

(10000, 8)

In [45]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [46]:
df_original.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


Estas primeras salidas nos permiten recuperar el estado inicial del dataset: cuántas filas y columnas tiene, qué tipos de datos asignó Pandas y cuántos valores faltantes reconoce en cada columna.

A partir de este diagnóstico inicial vamos a crear una copia de trabajo y comenzar el proceso de limpieza.

## Crear una copia de trabajo

Después del diagnóstico inicial, vamos a crear una copia del dataset.

La copia nos permite aplicar transformaciones sin perder el archivo original cargado desde Kaggle. Esto es especialmente útil cuando construimos un flujo de limpieza, porque necesitamos poder comparar el estado inicial con el estado final.

Vamos a llamar `df_limpieza` a la copia de trabajo.

In [47]:
df_limpieza = df_original.copy()

df_limpieza.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


A partir de ahora, las transformaciones se aplicarán sobre `df_limpieza`.

El dataset original queda disponible en `df_original`, por si necesitamos revisar valores iniciales o comparar resultados.

## Paso 1: estandarizar nombres de columnas

El primer paso del flujo será estandarizar los nombres de columnas.

El dataset original tiene nombres como `Transaction ID`, `Price Per Unit`, `Payment Method` y `Transaction Date`. Estos nombres son claros, pero tienen espacios y mayúsculas. Para trabajar con mayor comodidad, vamos a convertirlos a un formato uniforme en minúsculas y con guiones bajos.

Este paso no cambia los datos. Solo mejora la estructura del `DataFrame` y facilita las transformaciones posteriores.

In [48]:
df_limpieza = df_limpieza.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})

df_limpieza.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

Ahora los nombres de columnas siguen un criterio más uniforme.

Este cambio nos permitirá escribir las próximas transformaciones con nombres más cómodos y consistentes.

## Paso 2: convertir columnas numéricas

Después de estandarizar los nombres, vamos a convertir las columnas que deberían ser numéricas.

En el diagnóstico inicial vimos que todas las columnas fueron cargadas como `object`. Esto incluye columnas que representan cantidades o importes:

```text
quantity
price_per_unit
total_spent
```

Antes de hacer cálculos con estas columnas, necesitamos convertirlas a formato numérico. Como el dataset contiene valores problemáticos, usaremos `pd.to_numeric()` con `errors="coerce"`.

Esto transformará en `NaN` los valores que no puedan convertirse, como `"UNKNOWN"` o `"ERROR"`.

In [49]:
columnas_numericas = [
    "quantity",
    "price_per_unit",
    "total_spent"
]

for columna in columnas_numericas:
    df_limpieza[columna] = pd.to_numeric(
        df_limpieza[columna],
        errors="coerce"
    )

df_limpieza[columnas_numericas].dtypes

,0
quantity,float64
price_per_unit,float64
total_spent,float64


Ahora las columnas seleccionadas quedaron en formato numérico.

Como usamos `errors="coerce"`, es importante revisar cuántos valores faltantes quedaron después de la conversión. Esta revisión nos permite ver el efecto de transformar valores problemáticos en `NaN`.

In [50]:
df_limpieza[columnas_numericas].isna().sum()

,0
quantity,479
price_per_unit,533
total_spent,502


Este conteo puede ser mayor que el conteo inicial de faltantes, porque ahora también se consideran faltantes los valores que estaban escritos como texto problemático y no pudieron convertirse a número.

Este paso deja preparadas las columnas numéricas para cálculos posteriores, como validar si el total registrado coincide con la cantidad multiplicada por el precio unitario.

## Paso 3: convertir la columna de fecha

Además de las columnas numéricas, necesitamos convertir la columna `transaction_date`.

En el diagnóstico inicial vimos que esta columna también fue cargada como `object`. Eso significa que Pandas la interpretó como texto, aunque visualmente sus valores parezcan fechas.

Para poder trabajar con información temporal, vamos a convertirla usando `pd.to_datetime()` con `errors="coerce"`.

In [51]:
df_limpieza["transaction_date"] = pd.to_datetime(
    df_limpieza["transaction_date"],
    errors="coerce"
)

df_limpieza["transaction_date"].dtype

dtype('<M8[ns]')

Ahora la columna `transaction_date` quedó en formato temporal.

Como usamos `errors="coerce"`, los valores que no pudieron interpretarse como fechas quedaron como `NaT`. Por eso, igual que hicimos con las columnas numéricas, conviene revisar cuántos valores faltantes temporales quedaron después de la conversión.

In [52]:
df_limpieza["transaction_date"].isna().sum()

np.int64(460)

Este conteo incluye los valores que ya estaban faltantes antes de convertir y también los valores textuales que no pudieron interpretarse como fechas.

A partir de esta conversión, el dataset queda preparado para crear columnas temporales o realizar análisis por fecha.

## Paso 4: revisar valores categóricos problemáticos

Después de convertir columnas numéricas y fechas, vamos a revisar algunas columnas categóricas.

En este dataset hay tres columnas categóricas importantes:

```text
item
payment_method
location
```

Estas columnas pueden contener valores válidos, valores faltantes y valores problemáticos como `"UNKNOWN"` o `"ERROR"`.

Antes de reemplazar nada, conviene revisar sus frecuencias.

In [53]:
columnas_categoricas = [
    "item",
    "payment_method",
    "location"
]

for columna in columnas_categoricas:
    print(f"\nColumna: {columna}")
    print("-" * 40)
    print(df_limpieza[columna].value_counts(dropna=False))


Columna: item
----------------------------------------
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

Columna: payment_method
----------------------------------------
payment_method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

Columna: location
----------------------------------------
location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


Esta revisión permite distinguir entre categorías válidas y valores problemáticos.

Por ejemplo, en `payment_method`, valores como `"Cash"`, `"Credit Card"` o `"Digital Wallet"` representan formas de pago reales. En cambio, `"UNKNOWN"` y `"ERROR"` no deberían interpretarse como métodos de pago normales.

Lo mismo ocurre con `item` y `location`. `"UNKNOWN"` puede indicar que el dato no se conoce, mientras que `"ERROR"` sugiere que hubo algún problema de carga o registro.

En este flujo vamos a tomar una decisión simple: convertir `"ERROR"` en `"UNKNOWN"` dentro de estas columnas categóricas. De esta manera agrupamos los valores no confiables en una misma categoría de dato desconocido.

Esta decisión no es universal. En otro contexto, podríamos conservar `"ERROR"` por separado para investigar fallas del sistema. Aquí lo hacemos para construir una versión más simple y consistente del dataset.

In [54]:
for columna in columnas_categoricas:
    df_limpieza[columna] = df_limpieza[columna].replace({
        "ERROR": "UNKNOWN"
    })

for columna in columnas_categoricas:
    print(f"\nColumna: {columna}")
    print("-" * 40)
    print(df_limpieza[columna].value_counts(dropna=False))


Columna: item
----------------------------------------
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      636
NaN          333
Name: count, dtype: int64

Columna: payment_method
----------------------------------------
payment_method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
UNKNOWN            599
Name: count, dtype: int64

Columna: location
----------------------------------------
location
NaN         3265
Takeaway    3022
In-store    3017
UNKNOWN      696
Name: count, dtype: int64


Después del reemplazo, `"ERROR"` ya no aparece como categoría separada en estas columnas. Sus casos quedaron agrupados dentro de `"UNKNOWN"`.

Todavía pueden quedar valores faltantes reales, representados como `NaN`. En este flujo no vamos a completar todos esos valores de manera automática. Preferimos conservarlos para no introducir información que no estaba en el dataset original.

La decisión tomada en este paso fue acotada: unificar valores categóricos problemáticos que cumplen una función similar dentro del análisis.

## Paso 5: crear columnas calculadas y de validación

Una vez que las columnas numéricas fueron convertidas, podemos crear nuevas variables útiles para el análisis.

En este dataset existe una relación esperada:

```text
total_spent = quantity × price_per_unit
```

La columna `total_spent` representa el total registrado en el archivo. Podemos crear una nueva columna, `calculated_total`, para calcular ese total a partir de `quantity` y `price_per_unit`.

Esta nueva columna nos permitirá validar si el total registrado coincide con el total que se obtiene mediante el cálculo.


In [55]:
df_limpieza["calculated_total"] = (
    df_limpieza["quantity"] * df_limpieza["price_per_unit"]
)

df_limpieza[
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total"
    ]
].head(10)

,quantity,price_per_unit,total_spent,calculated_total
0,2.0,2.0,4.0,4.0
1,4.0,3.0,12.0,12.0
2,4.0,1.0,NaN,4.0
3,2.0,5.0,10.0,10.0
4,2.0,2.0,4.0,4.0
5,5.0,4.0,20.0,20.0
6,3.0,3.0,9.0,9.0
7,4.0,4.0,16.0,16.0
8,5.0,3.0,15.0,15.0
9,5.0,4.0,20.0,20.0


Ahora tenemos dos valores relacionados:

```text
total_spent       → total registrado en el dataset
calculated_total  → total calculado a partir de quantity y price_per_unit
```

Para comparar estos valores, vamos a crear una columna de validación. Esta vez no vamos a usar una comparación booleana simple, porque si falta alguno de los datos necesarios, no podemos afirmar que haya una inconsistencia. En esos casos, simplemente no hay información suficiente para validar.

In [56]:
df_limpieza["estado_total"] = "sin_datos_suficientes"

condicion_datos_completos = (
    df_limpieza["quantity"].notna()
    & df_limpieza["price_per_unit"].notna()
    & df_limpieza["total_spent"].notna()
)

condicion_coincide = (
    condicion_datos_completos
    & (df_limpieza["total_spent"] == df_limpieza["calculated_total"])
)

condicion_no_coincide = (
    condicion_datos_completos
    & (df_limpieza["total_spent"] != df_limpieza["calculated_total"])
)

df_limpieza.loc[condicion_coincide, "estado_total"] = "coincide"
df_limpieza.loc[condicion_no_coincide, "estado_total"] = "no_coincide"

df_limpieza["estado_total"].value_counts(dropna=False)

,count
estado_total,
coincide,8544
sin_datos_suficientes,1456


La columna `estado_total` resume la validación de la relación entre cantidad, precio unitario y total registrado.

En este caso, la salida muestra dos estados:

```text
coincide
sin_datos_suficientes
````

No aparecen casos `no_coincide`. Esto indica que, entre las filas que tienen datos suficientes para comparar `quantity`, `price_per_unit` y `total_spent`, el total registrado coincide con el total calculado.

Las filas clasificadas como `sin_datos_suficientes` no indican una inconsistencia entre importes. Indican que falta alguno de los datos necesarios para hacer la validación.

En otros datasets, o incluso en otra versión de este mismo archivo, podría aparecer también el estado `no_coincide`. Si aparecieran muchos casos de ese tipo, deberíamos revisar posibles errores en `quantity`, `price_per_unit` o `total_spent`.

Crear columnas de validación ayuda a convertir una relación interna del dataset en una señal concreta de calidad de datos.

## Paso 6: crear columnas temporales

Como ya convertimos `transaction_date` a formato temporal, podemos crear columnas derivadas de esa fecha.

Estas columnas ayudan a preparar el dataset para análisis por períodos. Por ejemplo, podemos extraer el año, el mes y el día de la semana de cada transacción.

No estamos agregando información externa. Estamos reorganizando información que ya estaba dentro de la fecha original.

In [57]:
df_limpieza["year"] = df_limpieza["transaction_date"].dt.year
df_limpieza["month"] = df_limpieza["transaction_date"].dt.month
df_limpieza["day_of_week"] = df_limpieza["transaction_date"].dt.day_name()

df_limpieza[
    [
        "transaction_date",
        "year",
        "month",
        "day_of_week"
    ]
].head(10)

,transaction_date,year,month,day_of_week
0,2023-09-08,2023.0,9.0,Friday
1,2023-05-16,2023.0,5.0,Tuesday
2,2023-07-19,2023.0,7.0,Wednesday
3,2023-04-27,2023.0,4.0,Thursday
4,2023-06-11,2023.0,6.0,Sunday
5,2023-03-31,2023.0,3.0,Friday
6,2023-10-06,2023.0,10.0,Friday
7,2023-10-28,2023.0,10.0,Saturday
8,2023-07-28,2023.0,7.0,Friday
9,2023-12-31,2023.0,12.0,Sunday


Las columnas `year`, `month` y `day_of_week` permiten analizar las transacciones desde una perspectiva temporal.

Como vimos antes, es posible que `year` y `month` aparezcan con decimales, como `2023.0` o `9.0`. Esto ocurre porque algunas filas tienen fechas faltantes o no interpretables. Al derivar columnas desde una fecha faltante, Pandas necesita usar un formato que pueda representar esos valores ausentes.

También es posible que los nombres de los días aparezcan en inglés, como `Friday`, `Tuesday` o `Wednesday`. Esto depende del entorno y no afecta el análisis. Si más adelante necesitáramos presentar esos valores en español, podríamos aplicar un reemplazo o mapeo de nombres.

Estas columnas derivadas serán útiles para análisis posteriores. Por ejemplo, podríamos revisar ventas por mes, transacciones por día de la semana o períodos con mayor actividad.

## Paso 7: verificar el estado del DataFrame preparado

Después de aplicar varias transformaciones, conviene hacer una revisión general del `DataFrame`.

Hasta este punto realizamos varias acciones:

```text
creamos una copia de trabajo
estandarizamos nombres de columnas
convertimos columnas numéricas
convertimos la columna de fecha
unificamos valores categóricos problemáticos
creamos una columna calculada
creamos una columna de validación
creamos columnas temporales
```

Cada una de estas transformaciones modificó de alguna manera la estructura del dataset. Por eso, antes de continuar, necesitamos verificar cómo quedó.


In [58]:
df_limpieza.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,calculated_total,estado_total,year,month,day_of_week
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,4.0,coincide,2023.0,9.0,Friday
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,12.0,coincide,2023.0,5.0,Tuesday
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19,4.0,sin_datos_suficientes,2023.0,7.0,Wednesday
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27,10.0,coincide,2023.0,4.0,Thursday
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,4.0,coincide,2023.0,6.0,Sunday


In [59]:
df_limpieza.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9667 non-null   object        
 2   quantity          9521 non-null   float64       
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    7421 non-null   object        
 6   location          6735 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   calculated_total  9006 non-null   float64       
 9   estado_total      10000 non-null  object        
 10  year              9540 non-null   float64       
 11  month             9540 non-null   float64       
 12  day_of_week       9540 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(6)
memory usage: 1015.8+ KB


La salida de `info()` nos permite revisar los tipos de datos actuales y la cantidad de valores no nulos por columna.

Ahora deberíamos observar que las columnas numéricas ya no están cargadas como texto, y que `transaction_date` aparece como columna temporal.

También podemos revisar los faltantes actuales.

In [60]:
df_limpieza.isna().sum()

,0
transaction_id,0
item,333
quantity,479
price_per_unit,533
total_spent,502
payment_method,2579
location,3265
transaction_date,460
calculated_total,994
estado_total,0


Este conteo muestra los valores faltantes después de las transformaciones realizadas.

Es esperable que algunas columnas tengan más faltantes que en el diagnóstico inicial, especialmente las columnas numéricas y la columna de fecha. Esto ocurre porque valores como `"UNKNOWN"` o `"ERROR"` fueron convertidos a `NaN` o `NaT` durante la conversión de tipos.

Por ejemplo, `quantity` pasó de 138 faltantes iniciales a 479 después de la conversión; `price_per_unit`, de 179 a 533; `total_spent`, de 173 a 502; y `transaction_date`, de 159 a 460. Esa diferencia se explica por los valores textuales problemáticos que ahora quedaron representados como `NaN` o `NaT`.

Esto no significa necesariamente que el dataset haya empeorado. En realidad, ahora esos problemas quedaron representados de una manera más clara para Pandas, lo que facilita su análisis y tratamiento posterior.

## Paso 8: construir una versión final de trabajo

Después de aplicar y verificar las transformaciones principales, podemos construir una versión final del dataset preparado.

Esta versión no tiene por qué eliminar toda la información problemática. Su objetivo es dejar una tabla más clara, con nombres consistentes, tipos de datos adecuados y columnas útiles para el análisis.

Vamos a llamar `df_preparado` a esta versión.

In [61]:
columnas_finales = [
    "transaction_id",
    "item",
    "quantity",
    "price_per_unit",
    "total_spent",
    "calculated_total",
    "estado_total",
    "payment_method",
    "location",
    "transaction_date",
    "year",
    "month",
    "day_of_week"
]

df_preparado = df_limpieza[columnas_finales].copy()

df_preparado.head()

,transaction_id,item,quantity,price_per_unit,total_spent,calculated_total,estado_total,payment_method,location,transaction_date,year,month,day_of_week
0,TXN_1961373,Coffee,2.0,2.0,4.0,4.0,coincide,Credit Card,Takeaway,2023-09-08,2023.0,9.0,Friday
1,TXN_4977031,Cake,4.0,3.0,12.0,12.0,coincide,Cash,In-store,2023-05-16,2023.0,5.0,Tuesday
2,TXN_4271903,Cookie,4.0,1.0,NaN,4.0,sin_datos_suficientes,Credit Card,In-store,2023-07-19,2023.0,7.0,Wednesday
3,TXN_7034554,Salad,2.0,5.0,10.0,10.0,coincide,UNKNOWN,UNKNOWN,2023-04-27,2023.0,4.0,Thursday
4,TXN_3160411,Coffee,2.0,2.0,4.0,4.0,coincide,Digital Wallet,In-store,2023-06-11,2023.0,6.0,Sunday


El nuevo `DataFrame` conserva las columnas originales más importantes y agrega algunas columnas creadas durante el proceso de limpieza.

La columna `calculated_total` permite comparar el total calculado con el total registrado. La columna `estado_total` resume esa validación. Las columnas `year`, `month` y `day_of_week` permiten trabajar con la dimensión temporal del dataset.

También podemos revisar la estructura final.

In [62]:
df_preparado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9667 non-null   object        
 2   quantity          9521 non-null   float64       
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   calculated_total  9006 non-null   float64       
 6   estado_total      10000 non-null  object        
 7   payment_method    7421 non-null   object        
 8   location          6735 non-null   object        
 9   transaction_date  9540 non-null   datetime64[ns]
 10  year              9540 non-null   float64       
 11  month             9540 non-null   float64       
 12  day_of_week       9540 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(6)
memory usage: 1015.8+ KB


Esta versión preparada no representa una limpieza definitiva para cualquier objetivo posible. Todavía pueden quedar decisiones pendientes, especialmente sobre valores faltantes y categorías desconocidas.

Sin embargo, sí representa una mejora importante respecto del dataset crudo. Ahora tenemos nombres de columnas consistentes, tipos de datos más adecuados, valores categóricos problemáticos parcialmente unificados y columnas nuevas que ayudan a validar y analizar la información.

En un proyecto real, esta versión podría ser el punto de partida para análisis exploratorio, visualizaciones o nuevas transformaciones más específicas.

## Errores frecuentes en una limpieza encadenada

Cuando aplicamos varias transformaciones en secuencia, uno de los errores más frecuentes es perder de vista el orden del proceso.

Por ejemplo, no conviene crear una columna calculada antes de convertir las columnas necesarias a formato numérico. Si `quantity` o `price_per_unit` todavía están como texto, el cálculo de `calculated_total` puede fallar o producir resultados no confiables.

Otro error frecuente es modificar el dataset original sin conservar una copia. En este capítulo trabajamos con `df_original`, `df_limpieza` y `df_preparado`. Esta separación permite distinguir entre el punto de partida, la copia de trabajo y la versión preparada para continuar el análisis.

También debemos tener cuidado con los valores faltantes que aparecen después de convertir tipos. Al usar `errors="coerce"`, valores como `"UNKNOWN"` o `"ERROR"` pueden transformarse en `NaN` o `NaT`. Esto no significa que el problema haya aparecido recién en ese momento, sino que ahora Pandas lo representa de una manera más adecuada para el análisis.

Otro error posible es aplicar reemplazos categóricos sin revisar primero los valores existentes. En este capítulo reemplazamos `"ERROR"` por `"UNKNOWN"` solo después de revisar las frecuencias de `item`, `payment_method` y `location`.

También puede ser un problema crear muchas columnas nuevas sin un propósito claro. Las columnas agregadas deberían tener una función dentro del análisis o dentro del control de calidad. En este caso, `calculated_total` y `estado_total` sirven para validar la relación entre cantidad, precio unitario y total registrado. Las columnas `year`, `month` y `day_of_week` sirven para preparar análisis temporales.

Una buena rutina para una limpieza encadenada podría ser:

```text
diagnosticar el estado inicial
crear una copia de trabajo
estandarizar nombres
convertir tipos necesarios
revisar valores problemáticos
crear columnas útiles
verificar resultados intermedios
construir una versión final de trabajo
```

El objetivo de un flujo de limpieza no es aplicar muchas transformaciones, sino aplicar las transformaciones necesarias en un orden razonable y con verificaciones intermedias.


## Resumen del capítulo

En este capítulo integramos varias tareas de preparación de datos dentro de un flujo de limpieza paso a paso.

Hasta ahora habíamos trabajado muchos temas por separado: valores faltantes, duplicados, limpieza de textos, conversión de tipos, fechas, nombres de columnas y creación de nuevas variables. En este capítulo reunimos varias de esas herramientas en una secuencia más cercana a una situación real de trabajo.

Partimos del dataset original cargado desde Kaggle y realizamos un diagnóstico inicial. Revisamos sus dimensiones, su estructura, los tipos de datos asignados por Pandas y la cantidad de valores faltantes por columna.

Luego creamos una copia de trabajo:

```python
df_limpieza = df_original.copy()
```

Esto nos permitió conservar el dataset original y aplicar transformaciones sobre una versión editable.

El primer paso de limpieza fue estandarizar los nombres de columnas. Cambiamos nombres como `Transaction ID`, `Price Per Unit` y `Transaction Date` por nombres en minúsculas y con guiones bajos, como `transaction_id`, `price_per_unit` y `transaction_date`.

Después convertimos columnas numéricas usando `pd.to_numeric()` con `errors="coerce"`:

```python
for columna in columnas_numericas:
    df_limpieza[columna] = pd.to_numeric(
        df_limpieza[columna],
        errors="coerce"
    )
```

Esta conversión permitió transformar `quantity`, `price_per_unit` y `total_spent` en columnas numéricas. Los valores que no pudieron convertirse quedaron representados como `NaN`.

También convertimos la columna `transaction_date` usando `pd.to_datetime()`:

```python
df_limpieza["transaction_date"] = pd.to_datetime(
    df_limpieza["transaction_date"],
    errors="coerce"
)
```

Los valores que no pudieron interpretarse como fechas quedaron como `NaT`.

Luego revisamos columnas categóricas como `item`, `payment_method` y `location`. En esas columnas unificamos el valor `"ERROR"` dentro de `"UNKNOWN"`, tomando una decisión simple para agrupar valores problemáticos o no confiables.

Más adelante creamos una columna calculada:

```python
df_limpieza["calculated_total"] = (
    df_limpieza["quantity"] * df_limpieza["price_per_unit"]
)
```

Esta columna permitió comparar el total calculado con el total registrado en `total_spent`.

También creamos una columna de validación llamada `estado_total`, que distingue tres situaciones:

```text
coincide
no_coincide
sin_datos_suficientes
```

Esta columna resultó más informativa que una comparación booleana simple, porque permite diferenciar una inconsistencia real de una fila que no tiene datos suficientes para ser evaluada.

Después creamos columnas temporales derivadas de `transaction_date`:

```python
df_limpieza["year"] = df_limpieza["transaction_date"].dt.year
df_limpieza["month"] = df_limpieza["transaction_date"].dt.month
df_limpieza["day_of_week"] = df_limpieza["transaction_date"].dt.day_name()
```

Estas columnas preparan el dataset para futuros análisis por períodos.

Finalmente construimos una versión final de trabajo llamada `df_preparado`, seleccionando las columnas más relevantes para continuar el análisis.

La idea principal de este capítulo fue:

```text
Un flujo de limpieza organiza varias transformaciones en una secuencia razonable, verificable y orientada al análisis.
```

La limpieza encadenada no consiste en aplicar muchos comandos seguidos. Consiste en ordenar decisiones: diagnosticar, transformar, verificar y construir una versión del dataset más clara, consistente y útil.


## Próximo paso

El siguiente paso será trabajar con la validación posterior a la limpieza.

Vamos a revisar cómo evaluar si el dataset preparado quedó en buenas condiciones: tipos de datos, valores faltantes, categorías problemáticas, duplicados, rangos numéricos, fechas válidas, consistencia interna y dimensiones antes/después.

Ese capítulo funcionará como cierre técnico de esta etapa de preparación de datos.